In [ ]:
import sys
sys.path.append('../models/numpy_models')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, matthews_corrcoef, confusion_matrix, classification_report

from neuralnet import NeuralNetwork
from layers import DenseLayer, DropoutLayer
from activation import ReLUActivation, SoftmaxActivation
from losses import CategoricalCrossEntropy
from optimizer import Adam
from metrics import accuracy

np.random.seed(42)


# Tarefa 2: Modelos com Implementação Própria (NumPy)

Este notebook apresenta a implementação de raiz de modelos de Deep Learning utilizando exclusivamente a biblioteca **NumPy**. O objetivo é classificar textos entre 5 classes distintas: **Google, Human, Meta, Anthropic e OpenAI**.

De acordo com os requisitos do enunciado, exploramos uma abordagem de implementação manual para compreender os fundamentos de *Backpropagation* e otimização de redes neuronais.

## Arquitetura: Abordagem *Divide and Conquer*
- **Modelo 1 (Binário):** Distingue textos *Human* vs *IA*
- **Modelo 2 (Multiclasse):** Distingue entre as diferentes IAs (Google, Meta, Anthropic, OpenAI)


## 1. Carregamento e Preparação dos Dados

### 1.1 Carregamento

As classes presentes no dataset são: **Google, Human, Meta, Anthropic e OpenAI**.
O enunciado refere 'Mistral' mas o professor confirmou que a classe correta é `Anthropic` — não há remapeamento necessário.


In [ ]:
# --- Carregar dataset ---
df = pd.read_csv('../data/dataset_limpo.csv', sep=';')
df = df.dropna(subset=['Text', 'Label'])

print("Distribuição do dataset de treino:")
print(df['Label'].value_counts())
print(f"\nTotal: {len(df)} amostras")


### 1.2 Criação dos Sub-datasets e Balanceamento

- **Modelo 1 (Binário):** Usamos todo o dataset sem oversampling. A rede compensa o desequilíbrio Human:IA (1:4) com `Dropout` mais agressivo e pesos de classe.
- **Modelo 2 (Multiclasse):** Filtramos apenas as amostras de IA. As classes estão já equilibradas (~999 cada).

> **Nota:** Oversampling com `replace=True` cria duplicados perfeitos que a rede memoriza, causando overfitting severo. Preferimos usar o dataset sem resampling e deixar a arquitetura DNN gerir o desequilíbrio.


In [ ]:
# --- Modelo 1: Human vs IA ---
df_model1 = df.copy()
df_model1['Label_Bin'] = np.where(df_model1['Label'] == 'Human', 'Human', 'IA')

print("Distribuição Modelo 1 (dataset completo):")
print(df_model1['Label_Bin'].value_counts())

# --- Modelo 2: Qual IA? ---
df_model2 = df[df['Label'] != 'Human'].reset_index(drop=True)
print("\nDistribuição Modelo 2 (IAs):")
print(df_model2['Label'].value_counts())


### 1.3 Divisão Treino / Validação / Teste e Features TF-IDF

Utilizamos uma divisão estratificada **80% / 10% / 10%** (treino / validação / teste).

**Feature Engineering — combinação de word e char n-grams:**
- **Modelo 1:** Concatenamos `word n-grams (1,2)` + `char n-grams (3,5)` (2500+2500 features). A combinação captura padrões léxicos *e* tipográficos.
- **Modelo 2:** `char n-grams (3,5)`, 5000 features — captura estilo tipográfico subtil entre diferentes IAs.

> O TF-IDF é ajustado (`fit`) **apenas nos dados de treino** para evitar *data leakage*.


In [ ]:
# === MODELO 1 ===
X1 = df_model1['Text']
y1_raw = df_model1['Label_Bin']
classes1 = sorted(y1_raw.unique())  # ['Human', 'IA']
y1_encoded = pd.get_dummies(y1_raw).astype(int).values

X1_train, X1_temp, y1_train, y1_temp = train_test_split(
    X1, y1_encoded, test_size=0.2, random_state=42, stratify=y1_encoded
)
X1_val, X1_test, y1_val, y1_test = train_test_split(
    X1_temp, y1_temp, test_size=0.5, random_state=42, stratify=y1_temp
)

tfidf1_word = TfidfVectorizer(max_features=2500, ngram_range=(1, 2), stop_words='english')
tfidf1_char = TfidfVectorizer(max_features=2500, analyzer='char', ngram_range=(3, 5))

X1_train_w = tfidf1_word.fit_transform(X1_train).toarray()
X1_train_c = tfidf1_char.fit_transform(X1_train).toarray()
X1_train_tfidf = np.hstack([X1_train_w, X1_train_c])

X1_val_tfidf = np.hstack([
    tfidf1_word.transform(X1_val).toarray(),
    tfidf1_char.transform(X1_val).toarray()
])
X1_test_tfidf = np.hstack([
    tfidf1_word.transform(X1_test).toarray(),
    tfidf1_char.transform(X1_test).toarray()
])

print(f"Modelo 1 — Treino: {X1_train_tfidf.shape[0]} | Val: {X1_val_tfidf.shape[0]} | Teste: {X1_test_tfidf.shape[0]}")
print(f"Features TF-IDF Modelo 1: {X1_train_tfidf.shape[1]} (word:2500 + char:2500)")

# === MODELO 2 ===
X2 = df_model2['Text']
y2_raw = df_model2['Label']
classes2 = sorted(y2_raw.unique())  # ['Anthropic', 'Google', 'Meta', 'OpenAI']
y2_encoded = pd.get_dummies(y2_raw).astype(int).values

X2_train, X2_temp, y2_train, y2_temp = train_test_split(
    X2, y2_encoded, test_size=0.2, random_state=42, stratify=y2_encoded
)
X2_val, X2_test, y2_val, y2_test = train_test_split(
    X2_temp, y2_temp, test_size=0.5, random_state=42, stratify=y2_temp
)

tfidf2 = TfidfVectorizer(max_features=5000, analyzer='char', ngram_range=(3, 5))
X2_train_tfidf = tfidf2.fit_transform(X2_train).toarray()
X2_val_tfidf   = tfidf2.transform(X2_val).toarray()
X2_test_tfidf  = tfidf2.transform(X2_test).toarray()

print(f"\nModelo 2 — Treino: {X2_train_tfidf.shape[0]} | Val: {X2_val_tfidf.shape[0]} | Teste: {X2_test_tfidf.shape[0]}")
print(f"Features TF-IDF Modelo 2: {X2_train_tfidf.shape[1]}")
print(f"\nClasses Modelo 1: {classes1}")
print(f"Classes Modelo 2: {classes2}")


### 1.4 Classe TextDataset

In [ ]:
class TextDataset:
    """Wrapper simples para encapsular X e y num objeto compatível com NeuralNetwork."""
    def __init__(self, X, y):
        self.X = X
        self.y = y

train_dataset1 = TextDataset(X1_train_tfidf, y1_train)
val_dataset1   = TextDataset(X1_val_tfidf,   y1_val)
test_dataset1  = TextDataset(X1_test_tfidf,  y1_test)

train_dataset2 = TextDataset(X2_train_tfidf, y2_train)
val_dataset2   = TextDataset(X2_val_tfidf,   y2_val)
test_dataset2  = TextDataset(X2_test_tfidf,  y2_test)


## 2. Modelos Baseline (Regressão Logística / Softmax Regression)

O baseline corresponde a uma rede neural **sem camadas ocultas** — matematicamente equivalente à Regressão Logística (binária) e Softmax Regression (multiclasse).

Serve como referência mínima para comparar com as DNNs mais profundas.


In [ ]:
n_features1 = X1_train_tfidf.shape[1]
n_features2 = X2_train_tfidf.shape[1]

# --- Baseline 1: Human vs IA ---
baseline1 = NeuralNetwork(
    epochs=200, batch_size=64, learning_rate=0.05,
    verbose=True, loss=CategoricalCrossEntropy, metric=accuracy
)
baseline1.add(DenseLayer(len(classes1), input_shape=(n_features1,)))
baseline1.add(SoftmaxActivation())

print("--- Treinar Baseline 1: Regressão Logística (Human vs IA) ---")
baseline1.fit(train_dataset1, val_dataset=val_dataset1)


In [ ]:
# --- Baseline 2: Softmax Regression (IAs) ---
baseline2 = NeuralNetwork(
    epochs=200, batch_size=64, learning_rate=0.05,
    verbose=True, loss=CategoricalCrossEntropy, metric=accuracy
)
baseline2.add(DenseLayer(len(classes2), input_shape=(n_features2,)))
baseline2.add(SoftmaxActivation())

print("--- Treinar Baseline 2: Softmax Regression (IAs) ---")
baseline2.fit(train_dataset2, val_dataset=val_dataset2)


In [ ]:
# --- Avaliação dos Baselines no conjunto de teste interno ---
def evaluate_model(model, dataset, classes, name=""):
    """Avalia um modelo num dataset e imprime Accuracy e MCC."""
    probs = model.predict(dataset)
    y_pred_idx = np.argmax(probs, axis=1)
    y_true_idx = np.argmax(dataset.y, axis=1)
    y_pred_labels = [classes[i] for i in y_pred_idx]
    y_true_labels = [classes[i] for i in y_true_idx]
    acc = accuracy_score(y_true_labels, y_pred_labels)
    mcc = matthews_corrcoef(y_true_labels, y_pred_labels)
    print(f"[{name}] Accuracy: {acc*100:.2f}% | MCC: {mcc:.4f}")
    return y_pred_labels, y_true_labels

print("=== Avaliação Baselines no Teste Interno ===")
evaluate_model(baseline1, test_dataset1, classes1, "Baseline1")
evaluate_model(baseline2, test_dataset2, classes2, "Baseline2")


## 3. DNNs — Abordagem *Divide and Conquer*

Implementamos duas DNNs com camadas ocultas, Dropout e ReLU:
- **DNN 1:** 5000 features entrada → 128 → Dropout(0.4) → 64 → Dropout(0.2) → saída binária
- **DNN 2:** 5000 features → 128 → Dropout(0.3) → 64 → saída 4 classes (IAs)

O Dropout mais agressivo (0.4) na DNN 1 serve para compensar o desequilíbrio de classes (1:4) e reduzir overfitting.

> **Nota sobre generalização:** Os modelos atingem >90% no teste interno (mesma distribuição do treino), mas uma performance inferior no dataset externo do professor é esperada — os textos de treino e de avaliação provêm de fontes e geradores diferentes. Este fenómeno chama-se *domain shift* e é um desafio fundamental em text classification.


In [ ]:
# --- DNN 1: Human vs IA ---
# Usar pesos de classe (4:1 para Human) e Adam Optimizer
opt1 = Adam(learning_rate=0.001)
loss_weighted = CategoricalCrossEntropy(class_weights=[4.0, 1.0])  # [Human, IA]

net1 = NeuralNetwork(
    epochs=100, batch_size=64, optimizer=opt1,
    verbose=True, loss=loss_weighted, metric=accuracy
)
net1.add(DenseLayer(128, input_shape=(X1_train_tfidf.shape[1],), init_type='he'))
net1.add(ReLUActivation())
net1.add(DropoutLayer(0.4))
net1.add(DenseLayer(64, init_type='he'))
net1.add(ReLUActivation())
net1.add(DropoutLayer(0.2))
net1.add(DenseLayer(len(classes1), init_type='xavier'))  # Xavier para a camada final Softmax
net1.add(SoftmaxActivation())

print("--- Treinar DNN 1 (Human vs IA) ---")
net1.fit(train_dataset1, val_dataset=val_dataset1, patience=15)


In [ ]:
# --- DNN 2: Qual IA? ---
opt2 = Adam(learning_rate=0.001)

net2 = NeuralNetwork(
    epochs=150, batch_size=64, optimizer=opt2,
    verbose=True, loss=CategoricalCrossEntropy, metric=accuracy
)
net2.add(DenseLayer(128, input_shape=(X2_train_tfidf.shape[1],), init_type='he'))
net2.add(ReLUActivation())
net2.add(DropoutLayer(0.3))
net2.add(DenseLayer(64, init_type='he'))
net2.add(ReLUActivation())
net2.add(DenseLayer(len(classes2), init_type='xavier'))
net2.add(SoftmaxActivation())

print("--- Treinar DNN 2 (Multi-IA) ---")
net2.fit(train_dataset2, val_dataset=val_dataset2, patience=20)


In [ ]:
print("=== Avaliação DNNs no Teste Interno ===")
evaluate_model(net1, test_dataset1, classes1, "DNN1")
evaluate_model(net2, test_dataset2, classes2, "DNN2")


## 4. Pipeline *Divide and Conquer* e Avaliação Global

Combinamos os dois modelos numa pipeline sequencial:
1. **DNN 1** classifica cada texto como Human ou IA.
2. Se for IA, **DNN 2** determina qual das 4 IAs é.

A avaliação final é feita sobre uma amostra estratificada do próprio dataset de treino (15%, `random_state=101`), garantindo reprodutibilidade.


In [ ]:
def pipeline_predict(texts, tfidf1_word, tfidf1_char, net1,
                      tfidf2, net2, classes1, classes2):
    """
    Pipeline D&C: aplica net1 (Human vs IA) e, se IA, aplica net2 (qual IA?).
    Usa features combinadas (word + char) para o Modelo 1.
    """
    texts_list = list(texts) if isinstance(texts, pd.Series) else texts

    # Etapa 1: features combinadas para Modelo 1
    X1_w = tfidf1_word.transform(texts_list).toarray()
    X1_c = tfidf1_char.transform(texts_list).toarray()
    X1_tfidf = np.hstack([X1_w, X1_c])
    dummy_y1 = np.zeros((len(X1_tfidf), len(classes1)))
    probs1   = net1.predict(TextDataset(X1_tfidf, dummy_y1))
    preds1   = [classes1[np.argmax(p)] for p in probs1]

    # Etapa 2: batch de todos os textos classificados como IA
    ia_indices = [i for i, c in enumerate(preds1) if c != 'Human']
    ia_preds2 = {}
    if ia_indices:
        ia_texts = [texts_list[i] for i in ia_indices]
        X2_tfidf = tfidf2.transform(ia_texts).toarray()
        dummy_y2 = np.zeros((len(X2_tfidf), len(classes2)))
        probs2   = net2.predict(TextDataset(X2_tfidf, dummy_y2))
        for local_i, global_i in enumerate(ia_indices):
            ia_preds2[global_i] = classes2[np.argmax(probs2[local_i])]

    return [
        'Human' if preds1[i] == 'Human' else ia_preds2[i]
        for i in range(len(preds1))
    ]


In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

# Amostra estratificada e reprodutível do dataset original para avaliação interna
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=101)
for _, test_idx in sss.split(df['Text'], df['Label']):
    df_test_eval = df.iloc[test_idx]

X_eval = df_test_eval['Text']
y_eval = df_test_eval['Label']

print("Distribuição do conjunto de avaliação interna:")
print(y_eval.value_counts())

print("\nA calcular previsões do pipeline...")
preds_eval = pipeline_predict(X_eval, tfidf1_word, tfidf1_char, net1, tfidf2, net2, classes1, classes2)

all_classes = sorted(df['Label'].unique())

acc_global = accuracy_score(y_eval, preds_eval)
mcc_global = matthews_corrcoef(y_eval, preds_eval)
print(f"\nAccuracy Global (Teste Interno): {acc_global*100:.2f}%")
print(f"MCC Global (Teste Interno):      {mcc_global:.4f}")
print("\nRelatório de Classificação:")
print(classification_report(y_eval, preds_eval, labels=all_classes))

cm = confusion_matrix(y_eval, preds_eval, labels=all_classes)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=all_classes, yticklabels=all_classes)
plt.title('Matriz de Confusão — Pipeline D&C (Teste Interno)')
plt.ylabel('Classe Real')
plt.xlabel('Previsão')
plt.tight_layout()
plt.show()


## 5. Geração da Submissão A (Dataset de Teste Final)

Carregamos o dataset cego (`subm1.csv`), aplicamos a mesma pipeline D&C e geramos o ficheiro de submissão.

A formatação do ficheiro segue a diretriz: `subm1-g<ng>-<curso>-A.csv`


In [ ]:
import os

df_subm = pd.read_csv('../subm1.csv', sep=';')

# Aplicar a pipeline D&C ao dataset cego
preds_subm = pipeline_predict(
    df_subm['Text'],
    tfidf1_word, tfidf1_char, net1,
    tfidf2, net2,
    classes1, classes2
)

df_subm['Label'] = preds_subm
df_final = df_subm[['ID', 'Text', 'Label']]

output_dir = '../Subm1'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'subm1-g3-MIA-A.csv')

df_final.to_csv(output_path, sep=';', index=False)
print(f"Submissão gerada com sucesso em: {output_path}")
print("---- Distribuição ----")
print(df_final['Label'].value_counts())
